# Connect to the H5N1 Postgres database

A runnable smoke test for both connection paths. Full prose and troubleshooting live in
[`docs/DB_ACCESS.md`](../docs/DB_ACCESS.md).

**Getting in is two gates:** (1) an IAM-authenticated tunnel — your `@g.harvard.edu`
already has `cloudsql.client`; (2) the Postgres login — user `h5n1_app` + its password
from the team password manager (or your own IAM login, once IAM DB auth is enabled). The
public IP has no authorized networks, so the tunnel is the only way in.

Run **either** the *Local* section **or** the *Colab* section — not both.

## Local (Cloud SQL Auth Proxy)

First, in a **separate terminal**, start the proxy and leave it running:

```bash
cloud-sql-proxy harvard-capstone-499102:us-central1:h5n1-pg --port 5432
```

If you already run Postgres locally (it owns 5432), use `--port 5433` and set `DB_PORT=5433`
in your `.env` to match. Launch this notebook through the project env (`uv run jupyter lab`
or the Docker image) so the pinned deps and `.env` are on the path.

To browse tables visually instead, point pgAdmin / DBeaver / TablePlus at `127.0.0.1` on
the same port — see `docs/DB_ACCESS.md`.

In [ ]:
# The one way in for the whole codebase. Reads .env (DB_HOST/DB_PORT/DB_USER/...).
from h5n1.db import get_engine, check_connection

print(check_connection())  # -> "PostgreSQL 18.4 ..."  (raises if the tunnel/login is wrong)

In [ ]:
import pandas as pd

# Sanity-check row counts across the loaded facts.
pd.read_sql(
    """
    SELECT 'fact_h5n1_outbreak'       AS table, COUNT(*) AS rows FROM fact_h5n1_outbreak
    UNION ALL SELECT 'fact_wild_bird_detection', COUNT(*) FROM fact_wild_bird_detection
    UNION ALL SELECT 'fact_weather',             COUNT(*) FROM fact_weather
    UNION ALL SELECT 'dim_county',               COUNT(*) FROM dim_county
    UNION ALL SELECT 'dim_date',                 COUNT(*) FROM dim_date
    ORDER BY rows DESC
    """,
    get_engine(),
)

In [ ]:
# Example real query: top poultry-outbreak counties.
pd.read_sql(
    """
    SELECT c.state, c.county_name, SUM(f.birds_affected) AS birds
    FROM fact_h5n1_outbreak f
    JOIN dim_county c ON c.fips = f.fips
    GROUP BY c.state, c.county_name
    ORDER BY birds DESC NULLS LAST
    LIMIT 10
    """,
    get_engine(),
)

## Google Colab (Cloud SQL Python Connector)

Colab is a remote VM — no `localhost`, no local proxy. The Connector authenticates via
ADC and tunnels through the same IAM path (**no authorized networks opened**).

`auth.authenticate_user()` authenticates **the Google account signed into Colab** — it must
be your `@g.harvard.edu` (the one with `cloudsql.client`), not a personal Gmail. Do **not**
upload a service-account key JSON.

The cell below uses the shared `h5n1_app` password. Once IAM database authentication is
enabled (see [`docs/DB_ACCESS.md`](../docs/DB_ACCESS.md)), drop the password entirely: pass
`user="<you>@g.harvard.edu"` and `enable_iam_auth=True` to `connector.connect(...)`.

In [ ]:
# Colab only. Skip this whole section when running locally.
!pip install -q "cloud-sql-python-connector[pg8000]" sqlalchemy pandas

from google.colab import auth  # noqa: E402
auth.authenticate_user()  # sign in with your @g.harvard.edu account

In [ ]:
import getpass

import pandas as pd
import sqlalchemy
from google.cloud.sql.connector import Connector

DB_PASSWORD = getpass.getpass("h5n1_app password: ")  # prompt at runtime; never hardcode

connector = Connector()


def _getconn():
    return connector.connect(
        "harvard-capstone-499102:us-central1:h5n1-pg",
        "pg8000",
        user="h5n1_app",
        password=DB_PASSWORD,
        db="h5n1",
        # IAM DB auth alternative: user="<you>@g.harvard.edu", drop password,
        # add enable_iam_auth=True.
    )


engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=_getconn)

print(pd.read_sql("SELECT version()", engine).iloc[0, 0])
pd.read_sql("SELECT COUNT(*) AS outbreak_rows FROM fact_h5n1_outbreak", engine)